In [4]:
# ────────────────────────────────────────────────
# Simple Item-based Recommendation System (no extra install)
# ────────────────────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

print("=== Simple Item-based Recommendation System ===")

# Create user-item matrix
rating_matrix = master.pivot_table(
    index='UserId',
    columns='AttractionId',
    values='Rating',
    fill_value=0
)

print(f"User-Item matrix shape: {rating_matrix.shape}")

# Item-item similarity (cosine)
item_similarity = cosine_similarity(rating_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=rating_matrix.columns, columns=rating_matrix.columns)

# Recommendation function
def simple_recommend_for_user(user_id, n=5):
    if user_id not in rating_matrix.index:
        print(f"User {user_id} not found.")
        return []
    
    user_ratings = rating_matrix.loc[user_id]
    
    similar_scores = item_similarity_df.dot(user_ratings)
    
    already_rated = user_ratings[user_ratings > 0].index
    similar_scores = similar_scores[~similar_scores.index.isin(already_rated)]
    
    top_n = similar_scores.sort_values(ascending=False).head(n)
    
    results = []
    for attr_id, score in top_n.items():
        name = master[master['AttractionId'] == attr_id]['Attraction'].values
        name = name[0] if len(name) > 0 else f"ID {attr_id}"
        results.append((name, score))
    
    return results

# Example – most active user
example_user = rating_matrix.index[0]
print(f"\nTop 5 recommendations for User {example_user}:")
for name, score in simple_recommend_for_user(example_user, n=5):
    print(f" - {name} → Similarity score: {score:.3f}")

# Save similarity matrix
os.makedirs('models', exist_ok=True)
item_similarity_df.to_pickle('models/item_similarity.pkl')
print("\nItem similarity saved: models/item_similarity.pkl")

=== Simple Item-based Recommendation System ===
User-Item matrix shape: (33530, 30)

Top 5 recommendations for User 14:
 - Tegenungan Waterfall → Similarity score: 1.717
 - Tanah Lot Temple → Similarity score: 1.048
 - Uluwatu Temple → Similarity score: 0.952
 - Waterbom Bali → Similarity score: 0.779
 - Seminyak Beach → Similarity score: 0.755

Item similarity saved: models/item_similarity.pkl


In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)  # creates folder if missing

joblib.dump(rf, 'models/rf_rating_predictor.pkl')  # replace 'rf' with your actual variable name

print("Regression model saved")

In [3]:
import joblib
import os
import pandas as pd

# Create folder if missing
os.makedirs('models', exist_ok=True)
print("Models folder ready:", os.path.abspath('models'))

Models folder ready: C:\Users\NARKEES SALEEM\models


In [6]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os

print("Re-creating & saving recommendation similarity matrix...")

# Ensure master is loaded (if not, reload quickly)
# If master is not defined, add the load/recreate code from earlier messages

# Create user-item rating matrix
rating_matrix = master.pivot_table(
    index='UserId',
    columns='AttractionId',
    values='Rating',
    fill_value=0
)

print(f"Matrix shape: {rating_matrix.shape}")

# Compute item-item cosine similarity
item_similarity = cosine_similarity(rating_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=rating_matrix.columns,
    columns=rating_matrix.columns
)

# Save directly to models folder
os.makedirs('models', exist_ok=True)
item_similarity_df.to_pickle('models/item_similarity.pkl')

print("Similarity matrix saved successfully!")
print("File location: C:\\tourism_project\\models\\item_similarity.pkl")
print("Current models folder contents:")
print(os.listdir('models'))

Re-creating & saving recommendation similarity matrix...
Matrix shape: (33530, 30)
Similarity matrix saved successfully!
File location: C:\tourism_project\models\item_similarity.pkl
Current models folder contents:
['item_similarity.pkl', 'rf_rating_predictor.pkl', 'rf_visitmode_classifier.pkl', 'visitmode_label_encoder.pkl']


In [3]:
import pandas as pd

# Your data folder (change only if wrong)
DATA_FOLDER = r"C:\tourism_project\data"

df_trans     = pd.read_excel(f"{DATA_FOLDER}/Transaction.xlsx")
df_user      = pd.read_excel(f"{DATA_FOLDER}/User.xlsx")
df_item      = pd.read_excel(f"{DATA_FOLDER}/Item.xlsx")
df_type      = pd.read_excel(f"{DATA_FOLDER}/Type.xlsx")
df_mode      = pd.read_excel(f"{DATA_FOLDER}/Mode.xlsx")
df_city      = pd.read_excel(f"{DATA_FOLDER}/City.xlsx")
df_continent = pd.read_excel(f"{DATA_FOLDER}/Continent.xlsx")
df_country   = pd.read_excel(f"{DATA_FOLDER}/Country.xlsx")

print("All lookup tables loaded successfully")

All lookup tables loaded successfully


In [4]:
master = df_trans.merge(df_user, on='UserId', how='left') \
                 .merge(df_item, on='AttractionId', how='left') \
                 .merge(df_type, on='AttractionTypeId', how='left') \
                 .merge(df_mode, left_on='VisitMode', right_on='VisitModeId', how='left') \
                 .merge(df_city, left_on='AttractionCityId', right_on='CityId', how='left', suffixes=('', '_city'))

master = master.rename(columns={
    'VisitMode_x': 'VisitModeId_Num',
    'VisitMode_y': 'VisitMode_Name'
})

if 'VisitYear' in master.columns and 'VisitMonth' in master.columns:
    master['VisitYearMonth'] = master['VisitYear'].astype(str) + '-' + master['VisitMonth'].astype(str).str.zfill(2)

print("Master DataFrame recreated")
print("Shape:", master.shape)
print("Columns:", master.columns.tolist())

Master DataFrame recreated
Shape: (52930, 22)
Columns: ['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitModeId_Num', 'AttractionId', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'VisitModeId', 'VisitMode_Name', 'CityId_city', 'CityName', 'CountryId_city', 'VisitYearMonth']


In [7]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os

print("Re-creating & saving recommendation similarity matrix...")

# Ensure master is loaded (if not, reload quickly)
# If master is not defined, add the load/recreate code from earlier messages

# Create user-item rating matrix
rating_matrix = master.pivot_table(
    index='UserId',
    columns='AttractionId',
    values='Rating',
    fill_value=0
)

print(f"Matrix shape: {rating_matrix.shape}")

# Compute item-item cosine similarity
item_similarity = cosine_similarity(rating_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=rating_matrix.columns,
    columns=rating_matrix.columns
)

# Save directly to models folder
os.makedirs('models', exist_ok=True)
item_similarity_df.to_pickle('models/item_similarity.pkl')

print("Similarity matrix saved successfully!")
print("File location: C:\\tourism_project\\models\\item_similarity.pkl")
print("Current models folder contents:")
print(os.listdir('models'))

Re-creating & saving recommendation similarity matrix...
Matrix shape: (33530, 30)
Similarity matrix saved successfully!
File location: C:\tourism_project\models\item_similarity.pkl
Current models folder contents:
['item_similarity.pkl', 'rf_rating_predictor.pkl', 'rf_visitmode_classifier.pkl', 'visitmode_label_encoder.pkl']


In [10]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os

print("Creating and saving recommendation similarity matrix...")

# Ensure master is loaded (if not, add load/recreate code from earlier)
# Quick check
print("Master shape:", master.shape)

# Create user-item rating matrix
rating_matrix = master.pivot_table(
    index='UserId',
    columns='AttractionId',
    values='Rating',
    fill_value=0
)

print(f"Matrix created – shape: {rating_matrix.shape}")

# Compute item-item similarity
item_similarity = cosine_similarity(rating_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=rating_matrix.columns,
    columns=rating_matrix.columns
)

# Save with FULL ABSOLUTE PATH (no ambiguity)
save_path = r'C:\tourism_project\models\item_similarity.pkl'
item_similarity_df.to_pickle(save_path)

print(f"Saved successfully at: {save_path}")
print("Current models folder contents:")
print(os.listdir(r'C:\tourism_project\models'))

Creating and saving recommendation similarity matrix...
Master shape: (52930, 22)
Matrix created – shape: (33530, 30)
Saved successfully at: C:\tourism_project\models\item_similarity.pkl
Current models folder contents:
['item_similarity.pkl', 'rf_rating_predictor.pkl', 'rf_visitmode_classifier.pkl', 'visitmode_label_encoder.pkl']


In [11]:
import os
print("Current working directory:", os.getcwd())
print("Models folder contents:", os.listdir(r'C:\tourism_project\models'))

Current working directory: C:\Users\NARKEES SALEEM
Models folder contents: ['item_similarity.pkl', 'rf_rating_predictor.pkl', 'rf_visitmode_classifier.pkl', 'visitmode_label_encoder.pkl']


In [12]:
os.chdir(r'C:\tourism_project')
print("Changed working directory to:", os.getcwd())

Changed working directory to: C:\tourism_project
